# 05 — Sklearn Preprocessing & Pipelines

**Topics:** ColumnTransformer, Pipeline, custom transformers, train/test splits, preprocessing strategies.

**Reference:** [sklearn preprocessing](https://scikit-learn.org/stable/modules/preprocessing.html) | [Pipeline](https://scikit-learn.org/stable/modules/pipeline.html)


In [ ]:
import pandas as pd
import numpy as np
from sklearn.datasets import fetch_openml
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder, OrdinalEncoder, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import train_test_split

# Titanic: mixed types, missing values, classification target — ideal for pipeline exercises
titanic_raw = fetch_openml(name='titanic', version=1, as_frame=True, parser='auto').frame
titanic = titanic_raw[['pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked', 'survived']].copy()
titanic['survived'] = titanic['survived'].astype(int)
titanic['pclass'] = titanic['pclass'].astype(float)
print(titanic.shape)
print(titanic.isnull().sum())
titanic.head()

---
## Exercise 1 — Train/Test Split with Stratification

1. Split `titanic` into train/test sets: 80/20, stratified on `survived`, `random_state=42`.
2. Verify the class distribution is preserved (within 1% tolerance).
3. Separate features (`X_train`, `X_test`) from target (`y_train`, `y_test`).
4. Print the shape and class balance of each split.

In [ ]:
# YOUR CODE HERE
X_train, X_test, y_train, y_test = None, None, None, None

In [ ]:
# --- ASSERTIONS ---
assert X_train.shape[0] + X_test.shape[0] == len(titanic)
assert X_test.shape[0] / len(titanic) == pytest_approx(0.2, abs=0.01) if False else abs(X_test.shape[0] / len(titanic) - 0.2) < 0.01
train_rate = y_train.mean()
test_rate = y_test.mean()
assert abs(train_rate - test_rate) < 0.02, "Stratification must preserve class balance"
assert 'survived' not in X_train.columns, "Target must not be in features"
print(f"✓ Exercise 1 passed")
print(f"Train: {X_train.shape} | Test: {X_test.shape} | Train survival rate: {train_rate:.2%} | Test: {test_rate:.2%}")

---
## Exercise 2 — Imputation Strategies

1. For numeric columns (`age`, `fare`): impute with **median** using `SimpleImputer`.
2. For categorical columns (`sex`, `embarked`): impute with **most frequent** value.
3. **Only fit on `X_train`, transform both `X_train` and `X_test`** — no leakage.
4. Return `X_train_imputed` and `X_test_imputed` as DataFrames (same columns, same row order).
5. Verify zero nulls after imputation.

In [ ]:
def impute_data(X_train: pd.DataFrame, X_test: pd.DataFrame):
    """
    Fit imputers on X_train only, transform both.
    Returns (X_train_imputed, X_test_imputed) as DataFrames.
    """
    # YOUR CODE HERE
    pass

X_train_imputed, X_test_imputed = impute_data(X_train, X_test)

In [ ]:
# --- ASSERTIONS ---
assert X_train_imputed.isnull().sum().sum() == 0, "No nulls in train after imputation"
assert X_test_imputed.isnull().sum().sum() == 0, "No nulls in test after imputation"
assert list(X_train_imputed.columns) == list(X_train.columns)
assert X_train_imputed.shape == X_train.shape
assert X_test_imputed.shape == X_test.shape
print("✓ Exercise 2 passed")

---
## Exercise 3 — ColumnTransformer

**Task:** Build a `ColumnTransformer` that applies different preprocessing to different column types.

- Numeric columns (`age`, `fare`, `sibsp`, `parch`, `pclass`): median imputation → StandardScaler.
- Categorical columns (`sex`, `embarked`): most-frequent imputation → OneHotEncoder (handle_unknown='ignore', drop='first').

1. Build `preprocessor` using `ColumnTransformer`.
2. Fit on `X_train`, transform `X_train` and `X_test`.
3. Output arrays `X_train_processed` and `X_test_processed`.
4. Print the number of output features.

In [ ]:
def build_preprocessor():
    """
    Returns a fitted ColumnTransformer.
    """
    # YOUR CODE HERE
    pass

preprocessor = build_preprocessor()

# YOUR CODE HERE: fit and transform
X_train_processed = None
X_test_processed = None

In [ ]:
# --- ASSERTIONS ---
assert X_train_processed is not None
assert X_train_processed.shape[0] == X_train.shape[0]
assert X_test_processed.shape[0] == X_test.shape[0]
assert X_train_processed.shape[1] == X_test_processed.shape[1], "Same number of features in train and test"
assert not np.isnan(X_train_processed).any(), "No NaN in processed train"
assert not np.isnan(X_test_processed).any(), "No NaN in processed test"
print(f"✓ Exercise 3 passed — {X_train_processed.shape[1]} output features")

---
## Exercise 4 — Custom Transformer

**Task:** Build a custom sklearn-compatible transformer.

Create `FamilySizeTransformer` that:
1. Inherits from `BaseEstimator` and `TransformerMixin`.
2. In `transform()`, adds a new column `family_size = sibsp + parch + 1`.
3. Adds `is_alone = (family_size == 1).astype(int)`.
4. Drops `sibsp` and `parch` from the output.
5. The transformer must work inside a sklearn `Pipeline`.
6. Must have `fit()` that does nothing (just returns self) — stateless transformer.

In [ ]:
class FamilySizeTransformer(BaseEstimator, TransformerMixin):
    """
    Adds family_size and is_alone, drops sibsp and parch.
    """
    def fit(self, X, y=None):
        # YOUR CODE HERE
        pass

    def transform(self, X, y=None):
        # YOUR CODE HERE
        pass

In [ ]:
# --- ASSERTIONS ---
fst = FamilySizeTransformer()
result = fst.fit_transform(X_train)

assert isinstance(result, pd.DataFrame), "Must return DataFrame"
assert 'family_size' in result.columns
assert 'is_alone' in result.columns
assert 'sibsp' not in result.columns
assert 'parch' not in result.columns
assert result['is_alone'].isin([0, 1]).all()
assert (result['family_size'] >= 1).all()

# Test inside a pipeline
from sklearn.preprocessing import StandardScaler
pipe = Pipeline([('family', FamilySizeTransformer()), ('passthrough', 'passthrough')])
pipe.fit(X_train)
pipe_result = pipe.transform(X_test)
assert pipe_result.shape[0] == X_test.shape[0]
print("✓ Exercise 4 passed")

---
## Exercise 5 — Full Pipeline (Preprocessing + Model)

**Task:** Build an end-to-end pipeline that combines all preprocessing steps with a model.

1. Build `full_pipeline`: a `Pipeline` with steps:
   - `'family'`: `FamilySizeTransformer` (from Exercise 4)
   - `'preprocessor'`: `ColumnTransformer` that handles the modified column set after FamilySizeTransformer
   - `'classifier'`: `LogisticRegression(max_iter=1000, random_state=42)`
2. Fit on `X_train`, `y_train`.
3. Report `train_accuracy` and `test_accuracy`.
4. The pipeline must be cloneable: `sklearn.base.clone(full_pipeline)` must work.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.base import clone

def build_full_pipeline():
    """
    Returns unfitted full_pipeline with 3 steps.
    """
    # YOUR CODE HERE
    pass

full_pipeline = build_full_pipeline()

# YOUR CODE HERE: fit and evaluate
train_accuracy = None
test_accuracy = None

In [ ]:
# --- ASSERTIONS ---
assert isinstance(full_pipeline, Pipeline)
assert len(full_pipeline.steps) == 3
assert full_pipeline.steps[0][0] == 'family'
assert full_pipeline.steps[-1][0] == 'classifier'
assert 0.6 < train_accuracy <= 1.0, "Train accuracy should be reasonable"
assert 0.6 < test_accuracy <= 1.0, "Test accuracy should be reasonable"

cloned = clone(full_pipeline)
assert not hasattr(cloned.named_steps['classifier'], 'coef_'), "Cloned pipeline must be unfitted"

print(f"✓ Exercise 5 passed — Train: {train_accuracy:.2%} | Test: {test_accuracy:.2%}")

---
## Exercise 6 — Pipeline Inspection & Feature Names

**Task:** Extract feature names and coefficients from a fitted pipeline — a critical debugging skill.

Using `full_pipeline` from Exercise 5:

1. Extract the output feature names from the `preprocessor` step using `get_feature_names_out()`.
2. Extract the `LogisticRegression` coefficients.
3. Build a DataFrame `feature_importance` with columns `feature` and `coefficient`, sorted by `abs(coefficient)` descending.
4. Identify the top 3 most influential features.

In [ ]:
def extract_feature_importance(pipeline: Pipeline) -> pd.DataFrame:
    """
    Returns feature_importance DataFrame: feature, coefficient.
    Sorted by abs(coefficient) descending.
    """
    # YOUR CODE HERE
    pass

feature_importance = extract_feature_importance(full_pipeline)

In [ ]:
# --- ASSERTIONS ---
assert list(feature_importance.columns) == ['feature', 'coefficient']
assert feature_importance['coefficient'].abs().is_monotonic_decreasing, "Must be sorted by abs(coef)"
assert len(feature_importance) > 0
print(f"✓ Exercise 6 passed")
print("Top 5 features:")
print(feature_importance.head())

---
## Exercise 7 — Comparing Preprocessing Strategies

**Task:** A pipeline-level A/B test. Compare 3 preprocessing strategies for the same model.

Build 3 pipelines using `LogisticRegression(max_iter=1000)` with these numeric scalers:
- Pipeline A: `StandardScaler`
- Pipeline B: `MinMaxScaler`
- Pipeline C: No scaling (passthrough)

All use the same imputation and OHE. Evaluate each using 5-fold stratified cross-validation.
Return `comparison_df`: DataFrame with columns `pipeline`, `cv_mean_accuracy`, `cv_std_accuracy`, sorted by `cv_mean_accuracy` descending.

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

def compare_preprocessing_strategies(X: pd.DataFrame, y: pd.Series) -> pd.DataFrame:
    """
    Compare 3 preprocessing strategies via 5-fold CV.
    Returns comparison_df.
    """
    # YOUR CODE HERE
    pass

comparison_df = compare_preprocessing_strategies(X_train, y_train)

In [ ]:
# --- ASSERTIONS ---
assert list(comparison_df.columns) == ['pipeline', 'cv_mean_accuracy', 'cv_std_accuracy']
assert len(comparison_df) == 3
assert set(comparison_df['pipeline']) == {'StandardScaler', 'MinMaxScaler', 'NoScaling'}
assert comparison_df['cv_mean_accuracy'].is_monotonic_decreasing
assert (comparison_df['cv_mean_accuracy'] > 0.6).all(), "All pipelines should achieve > 60% accuracy"
print(f"✓ Exercise 7 passed")
print(comparison_df)